# **YOLO 모델 사용**

# 1.환경준비

## (1) 라이브러리 설치

In [ ]:
!pip install ultralytics -q

## (2) 라이브러리 불러오기

In [ ]:
from ultralytics import settings, YOLO
import matplotlib.pyplot as plt
import cv2
import os
from IPython.display import Video

## (3) YOLO 설정

* 파일 경로 설정

In [ ]:
# 현재 세팅을 확인해 봅시다.
settings

In [ ]:
# 콜랩 파일 탭에 보이는 경로('/content/')로 변경해 봅시다.
settings['datasets_dir'] = '/content/'
settings.update()
settings

# 2.모델 사용

## (1) 모델 다운로드

- 모델의 구조와 해당 구조에 맞게 사전 학습된 가중치를 불러온다.
- Parameters
    * model : 모델 구조 또는 모델 구조 + 가중치 설정. task와 맞는 모델을 선택해야 한다.
    * task : detect, segment, classify, pose 중 택일

In [ ]:
model = YOLO(model='yolo11n.pt', task='detect')

## (2) 모델 사용 : 이미지

아래 이미지에 대해서 객체 탐지를 해 봅시다.

![이미지](https://images.pexels.com/photos/139303/pexels-photo-139303.jpeg)


In [ ]:
image_path = 'https://images.pexels.com/photos/139303/pexels-photo-139303.jpeg'
results = model.predict(image_path, save=True)
results[0].show()  # 탐지된 객체 출력

## (3)실습
* 다양한 사진을 찾아서 object detection 해 봅시다.

## (4) 객체탐지 결과 열어보기

In [ ]:
results

In [ ]:
print(type(results))
print(len(results))

In [ ]:
type(results[0])

In [ ]:
results[0]

In [ ]:
results[0].names

In [ ]:
results[0].boxes

In [ ]:
for box in results[0].boxes:
    x_min, y_min, x_max, y_max = box.xyxy[0]  # 좌표
    conf = box.conf[0]
    cn = results[0].names[int(box.cls[0])]  # 클래스 이름
    print(f"좌표: {x_min}, {y_min}, {x_max}, {y_max} | conf. : {conf} | class : {cn}")

## (5) 모델사용 : 동영상

* sample.mp4 파일을 업로드 합니다.

In [ ]:
# colab 파일 업로드
from google.colab import files
uploaded = files.upload()

* 동영상 객체 탐지 실행

In [ ]:
# 동영상 객체 탐지 실행 및 결과 저장
results = model.predict("sample.mp4", save=True)  # 결과 자동 저장

In [ ]:
# 탐지된 동영상 결과 경로 확인
os.listdir("runs/detect/predict/")

* 콜랩에서 영상 play를 위한 세팅

In [ ]:
# 영상 코덱 설치
!apt-get install -y ffmpeg

In [ ]:
# AVI to MP4로 변환 (YOLO 탐지 결과 파일명에 맞게 수정)
input_video_path = "runs/detect/predict/sample.avi"  # YOLO 탐지 결과 파일명
output_video_path = "runs/detect/predict/video_converted.mp4"

# FFmpeg를 사용하여 변환 (코덱: libx264)
!ffmpeg -i {input_video_path} -vcodec libx264 {output_video_path}

* 영상 paly

In [ ]:
# YOLO 탐지 결과 동영상 재생
Video("runs/detect/predict/video_converted.mp4", embed=True)

# 3.Confidence Score, IoU

## (1) 이미지 탐지 : default

In [ ]:
# 기본 설정(conf=0.25, iou=0.45)
image_path = 'https://health.chosun.com/site/data/img_dir/2018/01/17/2018011700908_0.jpg'
results = model.predict(image_path, save=True)
results[0].show()  # 탐지된 객체 출력

## (2) Confidence 값 조정

In [ ]:
high_conf = model(image_path, conf=0.7)  # 신뢰도 증가
low_conf = model(image_path, conf=0.1)   # 신뢰도 감소

* 탐색 결과

In [ ]:
# Matplotlib 서브플롯 설정
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# 결과를 NumPy 배열로 변환하여 Matplotlib에 표시 (BGR → RGB 변환 포함)
axes[0].imshow(cv2.cvtColor(high_conf[0].plot(), cv2.COLOR_BGR2RGB))  # 높은 신뢰도
axes[0].set_title("High Confidence (conf=0.7)")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(low_conf[0].plot(), cv2.COLOR_BGR2RGB))  # 낮은 신뢰도
axes[1].set_title("Low Confidence (conf=0.1)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## (3) IoU 임계 값 조정

In [ ]:
high_iou = model(image_path, iou=0.7)   # IoU 증가
low_iou = model(image_path, iou=0.01)    # IoU 감소

In [ ]:
# Matplotlib 서브플롯 설정
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# 결과를 NumPy 배열로 변환하여 Matplotlib에 표시 (BGR → RGB 변환 포함)
axes[0].imshow(cv2.cvtColor(high_iou[0].plot(), cv2.COLOR_BGR2RGB))  # 높은 IoU
axes[0].set_title("High IoU (iou=0.7)")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(low_iou[0].plot(), cv2.COLOR_BGR2RGB))  # 낮은 IoU
axes[1].set_title("Low IoU (iou=0.1)")
axes[1].axis("off")

plt.tight_layout()
plt.show()